# Final Merged Cash + Crypto Intelligence Pipeline

**CASHNET — Unified Fraud Intelligence System**

This notebook combines two independently trained models — a **Cash / Traditional Fraud model** and a **Crypto model** — into a single inference pipeline.

Given a new cyber complaint (banking details, a crypto wallet, or both), the pipeline:
1. Routes the input to the correct model(s)
2. Collects each model's risk score and specialised output (hotspots / wallet attribution)
3. Merges both into a single **Combined Risk Level** and an **actionable recommendation**

> This notebook does **not** train any models. It only loads previously trained `cash.pkl` and `crypto.pkl` pipelines and wires them together.

---

**Sections**
1. Setup & Drive Mount
2. Load Saved Models (`cash.pkl` and `crypto.pkl`)
3. Define Input Schema
4. Cash Model Inference Function
5. Crypto Model Inference Function
6. Merged Pipeline Logic
7. Final `generate_intelligence()` Function
8. Example Usage & Test Cases
9. Summary & How to Use


## 1. Setup & Drive Mount

Mount Google Drive and set up the base configuration. Update `MODEL_DIR` below if your models live somewhere else — everything downstream reads from this single variable.


In [ ]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- Imports ---
import os
import json
import pickle
import warnings
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")


In [ ]:
# --- Configuration: edit this if your paths differ ---

# Base folder where trained models are stored on Drive
MODEL_DIR = "/content/drive/MyDrive/CASHNET/models/"

# Accepted filenames for each model (first one found wins)
CASH_MODEL_CANDIDATES = ["cash.pkl", "cash_pipeline.pkl"]
CRYPTO_MODEL_CANDIDATES = ["crypto.pkl", "crypto_pipeline.pkl"]

# Risk thresholds used across the pipeline (0-1 scale). Tune these to taste.
RISK_THRESHOLDS = {
    "low_max": 0.40,     # score <= this  -> Low
    "medium_max": 0.70,  # score <= this  -> Medium, above -> High
}

print(f"Model directory set to: {MODEL_DIR}")


## 2. Load Saved Models (`cash.pkl` and `crypto.pkl`)

Both models are loaded once, at notebook start-up, and cached in memory. `resolve_model_path()` checks each candidate filename so either naming convention (`cash.pkl` / `cash_pipeline.pkl`) works without editing code.

If a model file is missing, the pipeline will still run — it will simply skip that model's branch and note it in the output, rather than crashing.


In [ ]:
def resolve_model_path(base_dir: str, candidates: List[str]) -> Optional[str]:
    """Return the first existing path among candidate filenames, or None."""
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.exists(path):
            return path
    return None


def load_model(path: Optional[str], label: str):
    """Safely unpickle a model, returning None (with a warning) on failure."""
    if path is None:
        print(f"[WARN] {label} model file not found. {label} branch will be disabled.")
        return None
    try:
        with open(path, "rb") as f:
            model = pickle.load(f)
        print(f"[OK] {label} model loaded from: {path}")
        return model
    except Exception as e:
        print(f"[ERROR] Failed to load {label} model from {path}: {e}")
        return None


cash_model_path = resolve_model_path(MODEL_DIR, CASH_MODEL_CANDIDATES)
crypto_model_path = resolve_model_path(MODEL_DIR, CRYPTO_MODEL_CANDIDATES)

cash_model = load_model(cash_model_path, "Cash")
crypto_model = load_model(crypto_model_path, "Crypto")

print("\nModel availability:")
print(f"  Cash model available:   {cash_model is not None}")
print(f"  Crypto model available: {crypto_model is not None}")


## 3. Define Input Schema

A single, flexible input dictionary covers three cases:

- **Cash-only** — only `banking_data` is supplied
- **Crypto-only** — only `crypto_data` is supplied
- **Both** — both are supplied (e.g. a complaint that mentions both a bank account and a wallet)

Edit the field names inside `banking_data` / `crypto_data` to match the exact features your trained `cash.pkl` / `crypto.pkl` pipelines expect — the schema below is a sensible, commonly-used starting point (adjust in `prepare_cash_features()` / `prepare_crypto_features()` in Sections 4 & 5 if your model expects different column names or ordering).


In [ ]:
# --- Input schema (dictionary-based) ---
#
# complaint_input = {
#     "complaint_id": "CMP-2026-00123",
#     "banking_data": {                      # optional - omit/None if not applicable
#         "transaction_amount": 45000,
#         "avg_monthly_transaction": 12000,
#         "num_transactions_last_30d": 18,
#         "account_age_days": 220,
#         "victim_bank": "SBI",
#         "suspect_bank": "HDFC",
#         "suspect_account_number": "XXXXXXXX1234",
#         "atm_withdrawal_flag": 1,
#         "location_lat": 20.2961,
#         "location_lon": 85.8245,
#     },
#     "crypto_data": {                       # optional - omit/None if not applicable
#         "wallet_address": "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa",
#         "chain": "BTC",
#         "tx_amount": 0.45,
#         "num_hops_from_victim": 2,
#         "num_incoming_tx": 5,
#         "num_outgoing_tx": 3,
#         "wallet_age_days": 40,
#         "linked_to_known_mixer": 0,
#     },
# }

EXAMPLE_SCHEMA = {
    "complaint_id": "string (unique complaint / case ID)",
    "banking_data": {
        "transaction_amount": "float - amount involved in the disputed transaction",
        "avg_monthly_transaction": "float - suspect/victim account's typical monthly volume",
        "num_transactions_last_30d": "int",
        "account_age_days": "int",
        "victim_bank": "string",
        "suspect_bank": "string",
        "suspect_account_number": "string (masked)",
        "atm_withdrawal_flag": "int (0/1) - whether funds were withdrawn via ATM",
        "location_lat": "float - last known withdrawal/branch latitude",
        "location_lon": "float - last known withdrawal/branch longitude",
    },
    "crypto_data": {
        "wallet_address": "string",
        "chain": "string - e.g. BTC, ETH, USDT-TRC20",
        "tx_amount": "float",
        "num_hops_from_victim": "int",
        "num_incoming_tx": "int",
        "num_outgoing_tx": "int",
        "wallet_age_days": "int",
        "linked_to_known_mixer": "int (0/1)",
    },
}

print(json.dumps(EXAMPLE_SCHEMA, indent=2))


## 4. Cash Model Inference Function

Converts `banking_data` into the feature vector the Cash model expects, runs inference, and returns a small structured result: risk score + predicted high-risk hotspot(s).

`predict_proba` is used when available (for a probability-style risk score); otherwise falls back to `predict` / `decision_function`.


In [ ]:
def prepare_cash_features(banking_data: Dict[str, Any]) -> pd.DataFrame:
    """
    Convert the raw banking_data dict into the feature DataFrame expected by
    the Cash model. Adjust column names/order here to match how cash.pkl
    was trained.
    """
    row = {
        "transaction_amount": banking_data.get("transaction_amount", 0),
        "avg_monthly_transaction": banking_data.get("avg_monthly_transaction", 0),
        "num_transactions_last_30d": banking_data.get("num_transactions_last_30d", 0),
        "account_age_days": banking_data.get("account_age_days", 0),
        "atm_withdrawal_flag": banking_data.get("atm_withdrawal_flag", 0),
    }
    return pd.DataFrame([row])


def predict_cash_hotspot(banking_data: Dict[str, Any]) -> str:
    """
    Placeholder hotspot lookup. If your cash.pkl bundle already includes a
    hotspot/geo model (e.g. clustering on lat/lon), call it here instead.
    Otherwise this derives a simple human-readable label from location +
    withdrawal behaviour, which you can replace with a real geo-model call.
    """
    lat = banking_data.get("location_lat")
    lon = banking_data.get("location_lon")
    bank = banking_data.get("suspect_bank", "Unknown Bank")

    if lat is not None and lon is not None:
        return f"Near ({lat:.4f}, {lon:.4f}) - {bank} branch/ATM cluster"
    return f"{bank} - exact ATM/branch cluster unavailable (no coordinates supplied)"


def run_cash_model(banking_data: Dict[str, Any]) -> Dict[str, Any]:
    """Run the Cash model on a single complaint's banking data."""
    if cash_model is None:
        return {
            "available": False,
            "risk_score": None,
            "predicted_hotspot": None,
            "note": "Cash model not loaded - branch skipped.",
        }

    features = prepare_cash_features(banking_data)

    try:
        if hasattr(cash_model, "predict_proba"):
            risk_score = float(cash_model.predict_proba(features)[:, 1][0])
        elif hasattr(cash_model, "decision_function"):
            raw = float(cash_model.decision_function(features)[0])
            risk_score = float(1 / (1 + np.exp(-raw)))  # squash to 0-1
        else:
            risk_score = float(cash_model.predict(features)[0])
    except Exception as e:
        return {
            "available": False,
            "risk_score": None,
            "predicted_hotspot": None,
            "note": f"Cash model inference failed: {e}",
        }

    hotspot = predict_cash_hotspot(banking_data)

    return {
        "available": True,
        "risk_score": round(risk_score, 4),
        "predicted_hotspot": hotspot,
        "note": None,
    }


## 5. Crypto Model Inference Function

Converts `crypto_data` into the feature vector the Crypto model expects, runs inference, and returns a risk score plus a wallet attribution / high-risk indicator.


In [ ]:
def prepare_crypto_features(crypto_data: Dict[str, Any]) -> pd.DataFrame:
    """
    Convert the raw crypto_data dict into the feature DataFrame expected by
    the Crypto model. Adjust column names/order here to match how
    crypto.pkl was trained.
    """
    row = {
        "tx_amount": crypto_data.get("tx_amount", 0),
        "num_hops_from_victim": crypto_data.get("num_hops_from_victim", 0),
        "num_incoming_tx": crypto_data.get("num_incoming_tx", 0),
        "num_outgoing_tx": crypto_data.get("num_outgoing_tx", 0),
        "wallet_age_days": crypto_data.get("wallet_age_days", 0),
        "linked_to_known_mixer": crypto_data.get("linked_to_known_mixer", 0),
    }
    return pd.DataFrame([row])


def attribute_wallet(crypto_data: Dict[str, Any], risk_score: float) -> str:
    """
    Placeholder wallet attribution / nearest-VASP lookup. If crypto.pkl's
    bundle includes a VASP-matching component or an external API/lookup
    table, call it here instead of this heuristic.
    """
    if crypto_data.get("linked_to_known_mixer"):
        return "High-risk: wallet linked to a known mixing/tumbling service"
    if risk_score >= RISK_THRESHOLDS["medium_max"]:
        return "High-risk cluster: no confirmed VASP match, exhibits layering pattern"
    if risk_score >= RISK_THRESHOLDS["low_max"]:
        return "Medium-risk: possible exchange deposit address, VASP attribution pending"
    return "Low-risk: consistent with legitimate exchange/VASP-linked wallet"


def run_crypto_model(crypto_data: Dict[str, Any]) -> Dict[str, Any]:
    """Run the Crypto model on a single complaint's wallet/transaction data."""
    if crypto_model is None:
        return {
            "available": False,
            "risk_score": None,
            "wallet_attribution": None,
            "note": "Crypto model not loaded - branch skipped.",
        }

    features = prepare_crypto_features(crypto_data)

    try:
        if hasattr(crypto_model, "predict_proba"):
            risk_score = float(crypto_model.predict_proba(features)[:, 1][0])
        elif hasattr(crypto_model, "decision_function"):
            raw = float(crypto_model.decision_function(features)[0])
            risk_score = float(1 / (1 + np.exp(-raw)))
        else:
            risk_score = float(crypto_model.predict(features)[0])
    except Exception as e:
        return {
            "available": False,
            "risk_score": None,
            "wallet_attribution": None,
            "note": f"Crypto model inference failed: {e}",
        }

    attribution = attribute_wallet(crypto_data, risk_score)

    return {
        "available": True,
        "risk_score": round(risk_score, 4),
        "wallet_attribution": attribution,
        "note": None,
    }


## 6. Merged Pipeline Logic

Routing rule:

| Input contains | Action |
|---|---|
| `banking_data` only | Run **Cash model** only |
| `crypto_data` only | Run **Crypto model** only |
| Both | Run **both models**, then merge |

The combined score is a simple weighted average when both branches are available (equal weight by default — tune `CASH_WEIGHT` / `CRYPTO_WEIGHT` if one model should dominate), or the single available score otherwise.


In [ ]:
# Weights used when both models produce a score (must sum to 1.0)
CASH_WEIGHT = 0.5
CRYPTO_WEIGHT = 0.5


def classify_risk_level(score: Optional[float]) -> str:
    """Map a 0-1 numeric score to a Low / Medium / High label."""
    if score is None:
        return "Unknown"
    if score <= RISK_THRESHOLDS["low_max"]:
        return "Low"
    if score <= RISK_THRESHOLDS["medium_max"]:
        return "Medium"
    return "High"


def route_and_run(input_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Decide which model(s) to run based on what's present in input_data,
    execute them, and return their raw results plus the routing decision.
    """
    banking_data = input_data.get("banking_data")
    crypto_data = input_data.get("crypto_data")

    has_cash = bool(banking_data)
    has_crypto = bool(crypto_data)

    if has_cash and has_crypto:
        route = "BOTH"
    elif has_cash:
        route = "CASH_ONLY"
    elif has_crypto:
        route = "CRYPTO_ONLY"
    else:
        route = "NONE"

    cash_result = run_cash_model(banking_data) if has_cash else None
    crypto_result = run_crypto_model(crypto_data) if has_crypto else None

    return {
        "route": route,
        "cash_result": cash_result,
        "crypto_result": crypto_result,
    }


def merge_results(routed: Dict[str, Any]) -> Dict[str, Any]:
    """Combine cash + crypto branch outputs into one overall risk level."""
    cash_result = routed["cash_result"]
    crypto_result = routed["crypto_result"]

    cash_score = cash_result["risk_score"] if cash_result and cash_result["available"] else None
    crypto_score = crypto_result["risk_score"] if crypto_result and crypto_result["available"] else None

    if cash_score is not None and crypto_score is not None:
        combined_score = round(CASH_WEIGHT * cash_score + CRYPTO_WEIGHT * crypto_score, 4)
    elif cash_score is not None:
        combined_score = cash_score
    elif crypto_score is not None:
        combined_score = crypto_score
    else:
        combined_score = None

    combined_level = classify_risk_level(combined_score)

    return {
        "combined_score": combined_score,
        "combined_level": combined_level,
    }


## 7. Final `generate_intelligence()` Function

The single public entry point for this notebook. Call `generate_intelligence(input_data)` with a complaint dictionary (see Section 3's schema) and get back a structured intelligence report.


In [ ]:
def build_recommendation(combined_level: str, routed: Dict[str, Any]) -> str:
    """Produce a short, actionable recommendation string for investigators."""
    cash_result = routed["cash_result"]
    crypto_result = routed["crypto_result"]

    notes = []

    if combined_level == "High":
        notes.append("Escalate immediately - flag for priority investigation.")
    elif combined_level == "Medium":
        notes.append("Flag for review within standard SLA; monitor for further activity.")
    elif combined_level == "Low":
        notes.append("Low priority - log for records, no immediate action required.")
    else:
        notes.append("Insufficient data to assess risk - request additional details from complainant.")

    if cash_result and cash_result.get("available") and cash_result.get("predicted_hotspot"):
        notes.append(f"Alert nearby ATMs/branches around: {cash_result['predicted_hotspot']}.")

    if crypto_result and crypto_result.get("available") and crypto_result.get("wallet_attribution"):
        notes.append(f"Wallet note: {crypto_result['wallet_attribution']}.")

    return " ".join(notes)


def generate_intelligence(input_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Main pipeline entry point.

    Parameters
    ----------
    input_data : dict
        Must follow the schema from Section 3. Requires at least one of
        'banking_data' or 'crypto_data'.

    Returns
    -------
    dict
        Structured final intelligence report, ready to print, log, or
        serialize to JSON.
    """
    routed = route_and_run(input_data)
    merged = merge_results(routed)
    recommendation = build_recommendation(merged["combined_level"], routed)

    cash_result = routed["cash_result"]
    crypto_result = routed["crypto_result"]

    report = {
        "complaint_id": input_data.get("complaint_id", "N/A"),
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "route_taken": routed["route"],
        "cash_analysis": {
            "risk_score": cash_result["risk_score"] if cash_result else None,
            "predicted_hotspot": cash_result["predicted_hotspot"] if cash_result else None,
            "note": cash_result["note"] if cash_result else "Not run - no banking_data supplied.",
        },
        "crypto_analysis": {
            "risk_score": crypto_result["risk_score"] if crypto_result else None,
            "wallet_attribution": crypto_result["wallet_attribution"] if crypto_result else None,
            "note": crypto_result["note"] if crypto_result else "Not run - no crypto_data supplied.",
        },
        "combined_risk_score": merged["combined_score"],
        "combined_risk_level": merged["combined_level"],
        "recommendation": recommendation,
    }

    return report


def print_report(report: Dict[str, Any]) -> None:
    """Pretty-print a generate_intelligence() report for human readability."""
    print("=" * 60)
    print(f"  FINAL INTELLIGENCE REPORT - {report['complaint_id']}")
    print("=" * 60)
    print(f"Generated At     : {report['generated_at']}")
    print(f"Route Taken      : {report['route_taken']}")
    print("-" * 60)
    print("CASH / TRADITIONAL FRAUD ANALYSIS")
    print(f"  Risk Score       : {report['cash_analysis']['risk_score']}")
    print(f"  Predicted Hotspot: {report['cash_analysis']['predicted_hotspot']}")
    if report['cash_analysis']['note']:
        print(f"  Note             : {report['cash_analysis']['note']}")
    print("-" * 60)
    print("CRYPTO ANALYSIS")
    print(f"  Risk Score        : {report['crypto_analysis']['risk_score']}")
    print(f"  Wallet Attribution: {report['crypto_analysis']['wallet_attribution']}")
    if report['crypto_analysis']['note']:
        print(f"  Note              : {report['crypto_analysis']['note']}")
    print("-" * 60)
    print(f"COMBINED RISK SCORE : {report['combined_risk_score']}")
    print(f"COMBINED RISK LEVEL : {report['combined_risk_level']}")
    print("-" * 60)
    print(f"RECOMMENDATION:\n  {report['recommendation']}")
    print("=" * 60)


## 8. Example Usage & Test Cases

Three scenarios below exercise all three routing paths: cash-only, crypto-only, and both.

> These use the real `cash_model` / `crypto_model` loaded in Section 2 if they were found on Drive; if a model file wasn't found, that branch will report `available: False` instead of crashing, so you can still see the full pipeline shape end-to-end.


In [ ]:
# --- Test Case 1: Cash-only complaint ---
test_case_1 = {
    "complaint_id": "CMP-2026-00101",
    "banking_data": {
        "transaction_amount": 85000,
        "avg_monthly_transaction": 15000,
        "num_transactions_last_30d": 22,
        "account_age_days": 95,
        "victim_bank": "SBI",
        "suspect_bank": "HDFC",
        "suspect_account_number": "XXXXXXXX7841",
        "atm_withdrawal_flag": 1,
        "location_lat": 20.2961,
        "location_lon": 85.8245,
    },
    "crypto_data": None,
}

report_1 = generate_intelligence(test_case_1)
print_report(report_1)


In [ ]:
# --- Test Case 2: Crypto-only complaint ---
test_case_2 = {
    "complaint_id": "CMP-2026-00102",
    "banking_data": None,
    "crypto_data": {
        "wallet_address": "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa",
        "chain": "BTC",
        "tx_amount": 0.75,
        "num_hops_from_victim": 3,
        "num_incoming_tx": 9,
        "num_outgoing_tx": 6,
        "wallet_age_days": 12,
        "linked_to_known_mixer": 1,
    },
}

report_2 = generate_intelligence(test_case_2)
print_report(report_2)


In [ ]:
# --- Test Case 3: Combined cash + crypto complaint ---
test_case_3 = {
    "complaint_id": "CMP-2026-00103",
    "banking_data": {
        "transaction_amount": 32000,
        "avg_monthly_transaction": 9000,
        "num_transactions_last_30d": 11,
        "account_age_days": 400,
        "victim_bank": "ICICI",
        "suspect_bank": "Axis",
        "suspect_account_number": "XXXXXXXX2290",
        "atm_withdrawal_flag": 0,
        "location_lat": 19.0760,
        "location_lon": 72.8777,
    },
    "crypto_data": {
        "wallet_address": "3J98t1WpEZ73CNmQviecrnyiWrnqRhWNLy",
        "chain": "ETH",
        "tx_amount": 1.2,
        "num_hops_from_victim": 1,
        "num_incoming_tx": 2,
        "num_outgoing_tx": 1,
        "wallet_age_days": 180,
        "linked_to_known_mixer": 0,
    },
}

report_3 = generate_intelligence(test_case_3)
print_report(report_3)


### Optional: Simple text-based report export

Writes a plain-text version of any report to disk — useful for attaching to a case file.


In [ ]:
def export_text_report(report: Dict[str, Any], out_dir: str = "/content") -> str:
    """Write a plain-text report file and return its path."""
    filename = f"intelligence_report_{report['complaint_id']}.txt"
    path = os.path.join(out_dir, filename)

    lines = [
        f"FINAL INTELLIGENCE REPORT - {report['complaint_id']}",
        f"Generated At: {report['generated_at']}",
        f"Route Taken : {report['route_taken']}",
        "",
        "CASH / TRADITIONAL FRAUD ANALYSIS",
        f"  Risk Score       : {report['cash_analysis']['risk_score']}",
        f"  Predicted Hotspot: {report['cash_analysis']['predicted_hotspot']}",
        "",
        "CRYPTO ANALYSIS",
        f"  Risk Score        : {report['crypto_analysis']['risk_score']}",
        f"  Wallet Attribution: {report['crypto_analysis']['wallet_attribution']}",
        "",
        f"COMBINED RISK SCORE: {report['combined_risk_score']}",
        f"COMBINED RISK LEVEL: {report['combined_risk_level']}",
        "",
        f"RECOMMENDATION: {report['recommendation']}",
    ]

    with open(path, "w") as f:
        f.write("\n".join(lines))

    print(f"Report written to: {path}")
    return path


# Example: export the combined test case report
export_text_report(report_3)


## 9. Summary & How to Use

**What this notebook does**
- Loads two pre-trained models (`cash.pkl`, `crypto.pkl`) from Google Drive — no retraining
- Accepts a single flexible input dictionary covering banking data, crypto data, or both
- Automatically routes to the right model(s) based on what's supplied
- Merges both models' outputs into one combined risk level and a plain-language recommendation

**How to use it in your own workflow**

1. Update `MODEL_DIR` in Section 1 if your `.pkl` files live somewhere other than `/content/drive/MyDrive/CASHNET/models/`.
2. If your models were trained on different feature names/order, adjust `prepare_cash_features()` (Section 4) and `prepare_crypto_features()` (Section 5) to match.
3. Call the pipeline for any new complaint:

```python
new_complaint = {
    "complaint_id": "CMP-2026-00500",
    "banking_data": { ... },   # or None
    "crypto_data": { ... },    # or None
}

report = generate_intelligence(new_complaint)
print_report(report)          # human-readable
# or
print(json.dumps(report, indent=2))   # JSON for downstream systems
```

4. To wire this into a larger dashboard or alerting system, `generate_intelligence()` returns a plain Python dict — pass it straight to your alerting/notification layer, a database write, or a heatmap visualization tool (e.g. plot `predicted_hotspot` coordinates with `folium`/`plotly` in a follow-up cell).

**Extending this notebook**
- Swap the heuristic `predict_cash_hotspot()` / `attribute_wallet()` placeholders for real sub-models or API lookups (e.g. an actual VASP-matching service) once available.
- Add a batch mode: loop `generate_intelligence()` over a DataFrame of complaints and export a single consolidated CSV/JSON of alerts.
- Add a heatmap cell using `folium` over `location_lat`/`location_lon` values across many cash-model reports to visualize hotspot clustering geographically.
